# E7 Notebook 27 — Overnight CIC-IDS2017 full source audits and verified GitHub autosync

**Purpose:** Unattended/resumable full audit of the eight CIC-IDS2017 TrafficLabelling original CSVs. It resumes two already-completed GitHub reports (**225,745 DDoS rows, 286,467 PortScan rows**) and then runs the remaining G0 schema scans **plus an extensive eight-file data-quality profiling pass**. Every completed safe aggregate report is immediately GitHub archived and verified.

**Fix for your actual error:** Original Notebook 26C's GitHub \`PUT\` of Friday PortScan succeeded, but its immediate \`GET\` returned HTTP 404. The new publisher checks GitHub's immutable object SHA, retries pending reads, records a durable upload receipt and **never blindly repeats PUT or overwrites** an existing file.

**Fix for prior Git problem:** This notebook uses **no git clone/pull/merge/reset** at all. It downloads a set of pinned-revision source files, validates each exact Git-blob hash and refuses to change any file in the source snapshot. It does not touch older Kaggle checkouts.

**Scientific safeguards:** No fitting, no official UNSW test access, no raw network rows/IP/identifiers published, no automatic +12 hours for apparently afternoon timestamps. The full G0 flags \`G0_NOT_CLEARED\` wherever ambiguities exist. External training is blocked until real time/flow-end/labels and splitting are validated; this overnight run produces substantial preregistered dataset evidence, **not invented performance results**.

**Requirements:** Kaggle with dataset \`dsshayan/dscicids\` mounted, Internet ON, Kaggle secret \`pushRare\` available for use. Before sleeping, run the first preflight cell and confirm \`PREFLIGHT PASSED\`, then use **Run All**. Kaggle may end sessions due to its runtime quotas; per-file GitHub results will survive interruptions.

## 1. Immutable source snapshot — no Git working-tree operations

Download ten version-pinned GitHub research source/config/test files to an E7-only Kaggle folder. Each is verified by the exact Git blob SHA published by GitHub. Existing unchanged files are reused; a modified/incorrect cached file causes an immediate halt. Run all synthetic uploader and source-audit tests **before** touching the actual dataset.

In [ ]:
from pathlib import Path
import sys, hashlib, subprocess, json
from urllib.request import urlopen

PINNED_COMMIT = '99a1782951ace3589a4b13a998e05411b30b80cc'
SOURCE_BLOBS = {
    "scripts/e7_g0_external_source_audit.py":  "67edf100fa438addeadefba17a8b9295fb733d54",
    "scripts/e7_g0_publish.py":  "0d7427d2eced1fba90c7b2b8ce29feadd443b22e",
    "scripts/e7_g0_publish_v2.py":  "cef711e90a865c15e336598017b223155ff04032",
    "scripts/e7_g0_extended_audit.py":  "7aced8bd3bd5bf55bd2c588538aca56e72f637ec",
    "configs/e7_g0_timestamp_registry_001.json":  "3e1c5f2e3252fa6923108eba02ac238b5a8561dc",
    "tests/test_e7_g0_external_source_audit.py":  "bf028eb5bdf6d4968403c608c79299c20c5709de",
    "tests/test_e7_g0_publish.py":  "dc7f52cce749d82e9b2dc12f63a3686303e97afe",
    "tests/test_e7_g0_timestamp_registry.py":  "55f70329719d48b2186179966f514c9992056718",
    "tests/test_e7_g0_publish_v2.py":  "99ec055ebfdcef8604db2b80bca5ae788032ce43",
    "tests/test_e7_g0_extended_audit.py":  "963c041e6869a1df765779ac69e6a18e22874049"
}
REPO_REMOTE='itsCodeBakery/rare-attack-spatiotemporal-ids'
RESEARCH_PREFIX='research/E7_CICIDS2017'
SOURCE_DIR=Path('/kaggle/working/E7_OVERNIGHT_001_IMMUTABLE_SOURCE')
STUDY=SOURCE_DIR/RESEARCH_PREFIX

def git_blob_sha(data):
    return hashlib.sha1(b'blob '+str(len(data)).encode()+b'\\0'+data).hexdigest()

# Use a normalized explicit NUL in the Git object header.
def verify_git_blob(data,expected):
    actual=hashlib.sha1(b'blob '+str(len(data)).encode()+bytes([0])+data).hexdigest()
    assert actual==expected,f'Wrong remote source revision: {actual} != {expected}'
    return True

for relative,blob_hash in SOURCE_BLOBS.items():
    local=STUDY/relative
    if local.exists():
        content=local.read_bytes()
        verify_git_blob(content,blob_hash)
    else:
        url=('https://raw.githubusercontent.com/'+REPO_REMOTE+'/'+PINNED_COMMIT+
             '/'+RESEARCH_PREFIX+'/'+relative)
        with urlopen(url,timeout=90) as response:
            content=response.read()
        verify_git_blob(content,blob_hash)
        local.parent.mkdir(parents=True,exist_ok=True)
        local.write_bytes(content)
print('PINNED SOURCE VERIFIED:',len(SOURCE_BLOBS),'files')
print('SOURCE COMMIT:',PINNED_COMMIT)
print('NO Git clone, merge, pull, reset or prior checkout modification')

test_paths=sorted((STUDY/'tests').glob('test_e7_g0_*.py'))
assert len(test_paths)==5,f'Expected 5 test modules, found {len(test_paths)}'
subprocess.run([sys.executable,'-m','pytest','-q',*map(str,test_paths)],check=True)
print('PREFLIGHT PASSED — all synthetic tests green; original research preserved')

## 2. Authenticate mounted external source and GitHub connection

This is a read-only source/secret preflight. Never print your GitHub secret. Confirm eight **exact** known CSV members beneath \`TrafficLabelling \` with its trailing space; each original timestamp format comes only from Notebook 26A's versioned registry. GitHub token is checked prior to any long scans.

In [ ]:
import importlib.util
from kaggle_secrets import UserSecretsClient
from urllib.error import HTTPError
import pandas as pd

def import_module(name,path):
    definition=importlib.util.spec_from_file_location(name,str(path))
    module=importlib.util.module_from_spec(definition)
    definition.loader.exec_module(module)
    return module

audit=import_module('e7_audit',STUDY/'scripts/e7_g0_external_source_audit.py')
publisher=import_module('e7_publisher_v2',STUDY/'scripts/e7_g0_publish_v2.py')
quality=import_module('e7_quality',STUDY/'scripts/e7_g0_extended_audit.py')
registry=json.loads((STUDY/'configs/e7_g0_timestamp_registry_001.json').read_text())
entries=registry['filename_registry']
DATA_ROOT=Path('/kaggle/input/datasets/dsshayan/dscicids')
assert DATA_ROOT.is_dir(), 'Attach the CIC-IDS2017 dataset before running.'
FILES=sorted(DATA_ROOT.rglob('*.csv'))
assert len(FILES)==8 and {p.name for p in FILES}==set(entries), 'External source file set changed.'
assert len({str(p.parent) for p in FILES})==1
assert FILES[0].parent.name=='TrafficLabelling ', 'Unexpected source directory.'
print('E7 SOURCE ROOT:',repr(str(FILES[0].parent)))

for p in FILES:
    sample=pd.read_csv(p,nrows=4,encoding='utf-8-sig',dtype='string',
                       usecols=lambda col:col.strip().lower()=='timestamp')
    assert sample.shape[1]==1, 'Missing original timestamp.'
    dates=pd.to_datetime(sample.iloc[:,0],format=entries[p.name]['timestamp_format'],
                         errors='coerce',exact=True)
    assert dates.notna().all(), 'Original timestamp sample fails registered parser.'
    assert set(dates.dt.day_name())=={entries[p.name]['expected_weekday']}

token=UserSecretsClient().get_secret('pushRare')
assert token, 'Set Kaggle pushRare secret before starting; never paste into outputs.'
publisher.api('GET','https://api.github.com/repos/'+REPO_REMOTE,token)
print('PREFLIGHT PASSED — dataset, timestamps and GitHub authentication available')
print('Existing earlier UNSW / M1 artifacts will not be read or changed.')

## 3. Overnight: resume and back up every G0 + extended per-file report

One code cell performs all eight G0 source/schema scans and all eight extended full-data scans. A source report is reused **only** when its exact original source SHA and timestamp format match (local checkpoint or original GitHub report). Each original G0 and extended G0 result is **verified on GitHub before proceeding**. Source bytes never change.

Extended profile covers: actual label support, parsed-day/clock distributions, numeric missing/non-finite/negative/zero features, constant columns, flow-duration raw range and anomalies, endpoint missingness, and **hash-candidate** duplicate or conflicting-label rows. It never uploads hashes, IP addresses, raw flow entries or predictions. A 64-bit hash collision is possible, so duplicate counts are explicitly descriptive, not exact evidence of leakage.

All detected afternoon 12-hour/24-hour ambiguity remains unresolved. The notebook never constructs purported causal five-flow context or launches IDS training while this validation gate is open.

In [ ]:
import base64
from collections import Counter

OLD_G0=Path('/kaggle/working/E7-EXT-VALIDITY-001/G0')
QUALITY_DIR=Path('/kaggle/working/E7-OVERNIGHT-001/G0_EXTENDED')
RECEIPTS=Path('/kaggle/working/E7-OVERNIGHT-001/UPLOAD_RECEIPTS')
OLD_G0.mkdir(parents=True,exist_ok=True)
QUALITY_DIR.mkdir(parents=True,exist_ok=True)
RECEIPTS.mkdir(parents=True,exist_ok=True)
report_pairs=[]

def safe_existing_public(stage,basename,expected_source_hash):
    filename=basename+('.audit.json' if stage=='G0' else '.quality.json')
    url=('https://api.github.com/repos/'+publisher.REPO+'/contents/'+
         publisher.PREFIX+'/'+stage+'/'+filename)
    try:
        record=publisher.api('GET',url,token)
    except HTTPError as exc:
        if exc.code==404:return None
        raise
    raw=base64.b64decode(record['content'])
    obj=json.loads(raw)
    publisher.encode_safe(obj,stage)
    source_field='input_artifact_sha256' if stage=='G0' else 'source_sha256'
    assert obj[source_field]==expected_source_hash, 'Uploaded original SHA differs from current CSV'
    assert (obj['external_source_basename'] if stage=='G0' else obj['source_file'])==basename
    return obj

def atomic_original_or_equal(target,obj):
    data=json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False)+'\n'
    if target.exists():
        original=json.loads(target.read_text(encoding='utf-8'))
        assert original==obj,'Local checkpoint differs: do not overwrite original.'
    else:
        tmp=target.with_suffix(target.suffix+'.new')
        tmp.write_text(data,encoding='utf-8')
        tmp.replace(target)

def registered_g0(path,source_sha):
    fmt=entries[path.name]['timestamp_format']
    o=audit.inspect(path,timestamp_format=fmt,max_rows=0,chunk_size=40000)
    assert o['input_artifact_sha256']==source_sha
    item=o['files'][0]
    known_weekday=entries[path.name]['expected_weekday']
    days={key.split(' | ',1)[0] for key in item['label_day_counts']}
    suspicious_days=[]
    for day in sorted(days):
        try:
            if pd.Timestamp(day).day_name()!=known_weekday:suspicious_days.append(day)
        except Exception:suspicious_days.append(day)
    if suspicious_days:
        o['explicit_followup_gates'].append(
            'Original date disagrees with expected filename weekday; review required.')
    if 'Afternoon' in path.name:
        o['explicit_followup_gates'].append(
            'CLOCK_AMPM_AMBIGUITY: Afternoon source has unqualified 1-11 hour in prior preview. '
            'The original clock convention has not been independently verified.')
    if suspicious_days or 'Afternoon' in path.name:
        item['status']='DATA_QUALITY_REVIEW_REQUIRED'
        o['status']='G0_NOT_CLEARED'
    return o

for number,p in enumerate(FILES,1):
    print('\n'+'='*67,flush=True)
    print(f'ORIGINAL E7 SOURCE [{number}/8] {p.name}',flush=True)
    source_sha=audit.fingerprint(p)
    base_path=OLD_G0/(p.name+'.audit.json')
    extended_path=QUALITY_DIR/(p.name+'.quality.json')

    if base_path.exists():
        basic=json.loads(base_path.read_text())
        assert basic['external_source_basename']==p.name
        assert basic['input_artifact_sha256']==source_sha
        assert basic['timestamp_format_declared']==entries[p.name]['timestamp_format']
        publisher.encode_safe(basic,'G0')
        print('REUSING source-checked existing original G0 checkpoint.',flush=True)
    else:
        basic=safe_existing_public('G0',p.name,source_sha)
        if basic is None:
            basic=registered_g0(p,source_sha)
            print('Completed full G0 original source scan.',flush=True)
        else:
            assert basic['timestamp_format_declared']==entries[p.name]['timestamp_format']
            print('RECOVERED exact original G0 result from GitHub.',flush=True)
        atomic_original_or_equal(base_path,basic)

    first=publisher.publish(base_path,token=token,stage='G0',
                            receipts_dir=RECEIPTS)
    print('GITHUB VERIFIED BASE:',first['status'],first['verified_via'],flush=True)

    if extended_path.exists():
        ext=json.loads(extended_path.read_text())
        assert ext['source_sha256']==source_sha
        publisher.encode_safe(ext,'G0_EXTENDED')
        print('REUSING exact source-checked extended checkpoint.',flush=True)
    else:
        ext=safe_existing_public('G0_EXTENDED',p.name,source_sha)
        if ext is None:
            print('SCANNING ALL ROWS for label, duration, feature,'
                  ' duplicate and clock quality...',flush=True)
            ext=quality.audit_file(
                p,timestamp_format=entries[p.name]['timestamp_format'],
                expected_weekday=entries[p.name]['expected_weekday'],chunk_size=35000)
            print('Extended original data-quality scan complete.',flush=True)
        else:
            print('RECOVERED original extended quality from GitHub.',flush=True)
        atomic_original_or_equal(extended_path,ext)

    second=publisher.publish(extended_path,token=token,
                             stage='G0_EXTENDED',receipts_dir=RECEIPTS)
    print('GITHUB VERIFIED EXTENDED:',second['status'],second['verified_via'],flush=True)
    assert basic['total_rows_inspected']==ext['rows'], 'Original vs extended row count drift'
    assert basic['files'][0]['labels']==ext['labels'], 'Original vs extended label counts drift'
    report_pairs.append((basic,ext))
    print('ROWS:',ext['rows'],
          '| Classes:',len(ext['labels']),
          '| Candidate duplicate rows:',ext['duplicates']['candidate_duplicate_rows'],
          '| Candidate contradictory labels:',
           ext['duplicates']['candidate_label_conflict_rows'])
    print('CLOCK AMBIGUOUS:',ext['timestamp_quality']['clock_ambiguous'],
          '| G0 STATUS:',basic['status'],flush=True)
    print('GITHUB RESEARCH REPORTS:',
          'https://github.com/'+publisher.REPO+'/tree/main/'+publisher.PREFIX)

print('\nE7 PER-SOURCE BACKUPS: ALL',len(report_pairs),'OF 8 ARCHIVED + VERIFIED.')
print('NO MODEL TRAINING. NO ORIGINAL UNSW TEST. TIME VALIDITY NOT CLEARED.')

## 4. Overnight aggregate report: original class support and provenance

Construct an aggregate-only cross-file summary from the **verified** source profiles. Publish immutably to GitHub and verify its actual content using the same corrected publisher. This is a **source/quality manuscript table**, *not* an externally validated intrusion-detection performance result.

In [ ]:
from collections import Counter
import time

assert len(report_pairs)==8, 'Finish all eight source audits before publishing summary.'
class_counts=Counter()
source_items=[]
for basic,extended in report_pairs:
    class_counts.update(extended['labels'])
    source_items.append({
       'file':extended['source_file'],
       'sha256':extended['source_sha256'],
       'rows':extended['rows'],
       'classes':len(extended['labels']),
       'g0_status':basic['status'],
       'clock_ambiguous':extended['timestamp_quality']['clock_ambiguous'],
       'timestamp_parse_failures':extended['timestamp_quality']['parse_failures'],
       'source_row_order_reversals':extended['timestamp_quality']['row_order_reversals'],
       'hash_candidate_duplicate_rows':extended['duplicates']['candidate_duplicate_rows'],
       'hash_candidate_conflicting_label_rows':extended['duplicates']['candidate_label_conflict_rows']})
source_items.sort(key=lambda obj:obj['file'])
summary={
  'schema':'E7_OVERNIGHT_FULL_SOURCE_SUMMARY_001',
  'stage':'READ_ONLY_ORIGINAL_CICIDS2017_G0_AND_EXTENDED',
  'run_id':'E7-OVERNIGHT-001',
  'source_code_pinned_commit':PINNED_COMMIT,
  'files_verified':len(report_pairs),
  'rows_total':sum(x['rows'] for x in source_items),
  'labels_total':dict(sorted(class_counts.items())),
  'sources':source_items,
  'all_g0_cleared':False,
  'causal_traffic_order_proven':False,
  'original_unsw_test_accessed':False,
  'model_training_executed':False,
  'scientific_status':'DATA_QUALITY_AND_CLOCK_SEMANTICS_UNRESOLVED',
  'limitations':[
    'Original CICIDS2017 labels are not independently corrected or validated.',
    'Afternoon source clock values lack AM/PM and may not be true 24-hour times.',
    'Flow Duration units and flow-start/end causality remain unverified.',
    'Candidate duplicates use 64-bit hash and cannot establish exact leakage.',
    'No chronological external model evaluation or novelty proof was performed.',
    'No comparison to publisher 2017 Wednesday within-dataset benchmark is claimed.'
  ]
}
summary_data=(json.dumps(summary,indent=2,sort_keys=True,ensure_ascii=False)+'\n').encode()
summary_path=Path('/kaggle/working/E7-OVERNIGHT-001/G0_OVERNIGHT_SUMMARY.json')
if summary_path.exists():
    assert summary_path.read_bytes()==summary_data, 'Original summary conflicts: refuse overwrite'
else:
    summary_path.parent.mkdir(parents=True,exist_ok=True)
    summary_path.write_bytes(summary_data)
remote_name=(publisher.PREFIX+'/G0_EXTENDED/E7_OVERNIGHT_001_SUMMARY.json')
url=('https://api.github.com/repos/'+publisher.REPO+'/contents/'+remote_name)
digest=hashlib.sha256(summary_data).hexdigest()
receipt_path=RECEIPTS/'OVERNIGHT_001_SUMMARY_RECEIPT.json'
stored=None
if receipt_path.exists():
    stored=json.loads(receipt_path.read_text())
    assert stored.get('local_sha256')==digest and stored.get('remote')==remote_name, \
        'Summary original upload receipt mismatch; do not overwrite.'
    blob=stored.get('blob_sha')
    commit=stored.get('commit_sha')
else:
    blob=commit=None

verified=None
for seconds in (0,1,2,3,5,8):
    if seconds:time.sleep(seconds)
    verified=publisher.check_remote(url,token,summary_data,blob=blob,commit=commit)
    if verified:break
if verified is None and stored is None:
    # Create-only! No update SHA, and no retry PUT on a 404.
    response=publisher.api('PUT',url,token,{
        'message':'research(e7): verified eight-file original source audit summary',
        'branch':'main','content':base64.b64encode(summary_data).decode('ascii')})
    blob=response['content']['sha']
    commit=response['commit']['sha']
    publisher.atomic_json(receipt_path,{'local_sha256':digest,'remote':remote_name,
                                        'blob_sha':blob,'commit_sha':commit,'status':'PENDING_VERIFY'})
    for seconds in (0,1,2,3,5,8,13):
        if seconds:time.sleep(seconds)
        verified=publisher.check_remote(url,token,summary_data,blob=blob,commit=commit)
        if verified:break
if not verified:
    raise RuntimeError('GitHub summary may already be uploaded. Receipt saved. Stop; no blind PUT.')
publisher.atomic_json(receipt_path,{'local_sha256':digest,'remote':remote_name,
    'blob_sha':blob or verified['blob_sha'],'commit_sha':commit,
    'status':'VERIFIED','verified_via':verified['via']})

print('\n'+'='*74)
print('OVERNIGHT E7 RESULTS BACKED UP AND GITHUB VERIFIED')
print('Original CSV files:',len(report_pairs),
      '| All source rows:',summary['rows_total'])
print('Class counts:',summary['labels_total'])
print('Source files requiring review:',
      sum(x['g0_status']=='G0_NOT_CLEARED' for x in source_items))
print('Ambiguous clock files:',
      [x['file'] for x in source_items if x['clock_ambiguous']])
print('SUMMARY URL: https://github.com/'+publisher.REPO+'/blob/main/'+remote_name)
print('NO IDS MODELS TRAINED. EXTERNAL VALIDATION / CAUSAL ORDER REMAIN PENDING.')
print('='*74)

## When you wake up

Look at the last cell's **SUMMARY URL** and the GitHub research directories:

- \`research/E7_CICIDS2017/results/G0/\` — eight original hashed class/schema/time scans (original DDoS and PortScan preserved).
- \`research/E7_CICIDS2017/results/G0_EXTENDED/\` — eight further comprehensive feature/duration/duplicate-quality scans and one manuscript-ready aggregate summary.
- \`/kaggle/working/E7-OVERNIGHT-001/UPLOAD_RECEIPTS/\` — private per-upload commit/blob verification receipts to support same-session recovery; no private credentials stored.

The next scientific step will be a *separate versioned notebook* to resolve actual AM/PM timing and duration/flow completion, review corrected CICIDS2017 labels, and freeze a leakage-aware external benchmark. We must not train a purported chronological IDS on this source while those gates are unresolved.

**Reproducibility warning:** Kaggle runtime may terminate overnight regardless of code. Every file that finishes either audit stage is synced immediately; rerunning this notebook can restore completed reports from public GitHub and continue without overwriting them.